<a href="https://colab.research.google.com/github/sasikumaar1006/Retinal_disease_detection/blob/main/notebooks/03_pre_trained_model_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets torch torchvision scikit-learn matplotlib numpy -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
ds = load_dataset("zacharielegault/Kermany2017-OCT")
print(ds)
train_data = ds["train"]
test_data = ds["test"]

print("Training samples:", len(train_data))
print("Testing samples:", len(test_data))

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 108309
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
})
Training samples: 108309
Testing samples: 1000


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Test/Validation transform
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class OCTDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image'].convert("RGB")
        label = item['label']

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

# Creating strictly separated datasets and data loaders to prevent data leakage
train_dataset = OCTDataset(train_data, transform=train_transform)
test_dataset = OCTDataset(test_data, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False) # shuffle=False for testing

print(f"Train DataLoader batches: {len(train_loader)}")
print(f"Test DataLoader batches: {len(test_loader)}")

Train DataLoader batches: 3385
Test DataLoader batches: 32


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def build_transfer_model(arch_name="resnet18", num_classes=4, freeze_backbone=False):
    arch_name = arch_name.lower()

    if arch_name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif arch_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        model.classifier[1] = nn.Linear(model.last_channel, num_classes)

    elif arch_name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)

    elif arch_name == "vgg16":
        model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_features, num_classes)

    else:
        raise ValueError(f"Unsupported architecture: {arch_name}")

    return model.to(device)

Using device: cuda


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / total
    acc = correct / total
    return avg_loss, acc, np.array(all_labels), np.array(all_preds)

In [ ]:
class_names = ["CNV", "DME", "DRUSEN", "NORMAL"]
model = build_transfer_model("resnet18", num_classes=4, freeze_backbone=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

epochs = 2  # Good starting run to compare against your 2-epoch Keras CNN
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, y_true, y_pred = evaluate_model(model, test_loader, criterion, device)
    print(
        f"Epoch {epoch}/{epochs} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Test Loss: {val_loss:.4f}, Test Acc: {val_acc:.4f}"
    )

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

Epoch 1/2 | Train Loss: 0.1369, Train Acc: 0.9538 | Test Loss: 0.1111, Test Acc: 0.9630
Epoch 2/2 | Train Loss: 0.0913, Train Acc: 0.9694 | Test Loss: 0.0877, Test Acc: 0.9730

Classification Report:
              precision    recall  f1-score   support

         CNV     0.9058    1.0000    0.9506       250
         DME     1.0000    1.0000    1.0000       250
      DRUSEN     1.0000    0.8920    0.9429       250
      NORMAL     0.9960    1.0000    0.9980       250

    accuracy                         0.9730      1000
   macro avg     0.9755    0.9730    0.9729      1000
weighted avg     0.9755    0.9730    0.9729      1000

Confusion Matrix:
 [[250   0   0   0]
 [  0 250   0   0]
 [ 26   0 223   1]
 [  0   0   0 250]]


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ["CNV", "DME", "DRUSEN", "NORMAL"]

# Load saved ResNet18 weights (skip if `model` is already in memory)
resnet18_cam_model = models.resnet18(weights=None)
resnet18_cam_model.fc = nn.Linear(resnet18_cam_model.fc.in_features, len(class_names))
resnet18_cam_model.load_state_dict(torch.load("resnet18_oct2017.pth", map_location=device))
resnet18_cam_model.to(device)
resnet18_cam_model.eval()

# Hook the last residual block in ResNet18
grad_cam = GradCAM(resnet18_cam_model, target_layer=resnet18_cam_model.layer4[-1])

imagenet_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
imagenet_std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def denormalize_image(tensor_chw):
    img = tensor_chw.cpu().numpy().transpose(1, 2, 0)
    img = (img * imagenet_std) + imagenet_mean
    return np.clip(img, 0.0, 1.0)

def plot_gradcam_grid(dataset, grad_cam_obj, indices, figsize=(14, 3.5)):
    n = len(indices)
    fig, axes = plt.subplots(n, 3, figsize=(13, 3.2 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, idx in enumerate(indices):
        img_tensor, true_idx = dataset[idx]
        input_batch = img_tensor.unsqueeze(0).to(device)

        cam, pred_idx, pred_prob = grad_cam_obj.generate(input_batch)
        img_rgb = denormalize_image(img_tensor)

        h, w, _ = img_rgb.shape
        cam_resized = cv2.resize(cam, (w, h), interpolation=cv2.INTER_LINEAR)
        heatmap_color = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
        heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB) / 255.0

        overlay = np.clip(0.6 * img_rgb + 0.4 * heatmap_color, 0.0, 1.0)

        true_label = class_names[true_idx]
        pred_label = class_names[pred_idx]

        axes[row, 0].imshow(img_rgb)
        axes[row, 0].set_title(f"Idx {idx} | True: {true_label}")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(cam_resized, cmap="jet")
        axes[row, 1].set_title(f"Grad-CAM ({pred_label}: {pred_prob:.2%})")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title("Overlay")
        axes[row, 2].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Collect first example per true class and up to 4 DRUSEN -> CNV misclassifications
seen_classes = set()
per_class_indices = []
drusen_as_cnv_indices = []

resnet18_cam_model.eval()
with torch.no_grad():
    for idx in range(len(test_dataset)):
        img_t, label_t = test_dataset[idx]
        if label_t not in seen_classes:
            seen_classes.add(label_t)
            per_class_indices.append(idx)

        pred_t = resnet18_cam_model(img_t.unsqueeze(0).to(device)).argmax(dim=1).item()
        if class_names[label_t] == "DRUSEN" and class_names[pred_t] == "CNV" and len(drusen_as_cnv_indices) < 4:
            drusen_as_cnv_indices.append(idx)

        if len(seen_classes) == len(class_names) and len(drusen_as_cnv_indices) >= 4:
            break

print("One sample per class indices:", per_class_indices)
plot_gradcam_grid(test_dataset, grad_cam, per_class_indices)

if drusen_as_cnv_indices:
    print("\nDRUSEN misclassified as CNV indices:", drusen_as_cnv_indices)
    plot_gradcam_grid(test_dataset, grad_cam, drusen_as_cnv_indices)

# Clean up hooks when finished
grad_cam.remove_hooks()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        self._fwd_handle = target_layer.register_forward_hook(self._save_activation)
        self._bwd_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        """
        input_tensor: shape (1, 3, H, W) on `device`
        class_idx: optional target class index; defaults to argmax prediction
        """
        self.model.eval()
        self.model.zero_grad()

        logits = self.model(input_tensor)
        probs = F.softmax(logits, dim=1)

        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()

        pred_prob = probs[0, class_idx].item()
        score = logits[0, class_idx]
        score.backward()

        # Channel weights via global average pooling of gradients
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        else:
            cam = np.zeros_like(cam)

        return cam, class_idx, pred_prob

    def remove_hooks(self):
        self._fwd_handle.remove()
        self._bwd_handle.remove()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ["CNV", "DME", "DRUSEN", "NORMAL"]


In [ ]:
torch.save(model.state_dict(), "resnet18_oct2017.pth")

# To load the resnet model
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loaded_model = models.resnet18(weights=None)
loaded_model.fc = nn.Linear(loaded_model.fc.in_features, 4)
loaded_model.load_state_dict(torch.load("resnet18_oct2017.pth", map_location=device))
loaded_model.to(device)
loaded_model.eval()

In [ ]:
import time
from google.colab import files

arch = "mobilenet_v2"
epochs = 2

print(f"Training {arch.upper()}...")
model = build_transfer_model(arch, num_classes=len(class_names), freeze_backbone=False)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

start_time = time.time()
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, y_true, y_pred = evaluate_model(model, test_loader, criterion, device)
    print(
        f"Epoch {epoch}/{epochs} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Test Loss: {val_loss:.4f}, Test Acc: {val_acc:.4f}",
        flush=True
    )

elapsed_min = (time.time() - start_time) / 60.0
print(f"\nCompleted {arch.upper()} in {elapsed_min:.1f} minutes.")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

weights_path = f"step5_{arch}_oct2017.pth"
torch.save(model.state_dict(), weights_path)
files.download(weights_path)

# Clear GPU memory before running the next model
del model, optimizer
torch.cuda.empty_cache()

Training MOBILENET_V2...
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 137MB/s]


Epoch 1/2 | Train Loss: 0.1556, Train Acc: 0.9483 | Test Loss: 0.2244, Test Acc: 0.9210
Epoch 2/2 | Train Loss: 0.0883, Train Acc: 0.9701 | Test Loss: 0.1660, Test Acc: 0.9450

Completed MOBILENET_V2 in 41.3 minutes.

Classification Report:
              precision    recall  f1-score   support

         CNV     0.8356    0.9960    0.9088       250
         DME     0.9920    0.9920    0.9920       250
      DRUSEN     0.9950    0.8000    0.8869       250
      NORMAL     0.9880    0.9920    0.9900       250

    accuracy                         0.9450      1000
   macro avg     0.9527    0.9450    0.9444      1000
weighted avg     0.9527    0.9450    0.9444      1000

Confusion Matrix:
 [[249   1   0   0]
 [  2 248   0   0]
 [ 47   0 200   3]
 [  0   1   1 248]]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import time
from google.colab import files

arch = "densenet121"
epochs = 2

print(f"Training {arch.upper()}...")
model = build_transfer_model(arch, num_classes=len(class_names), freeze_backbone=False)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

start_time = time.time()
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, y_true, y_pred = evaluate_model(model, test_loader, criterion, device)
    print(
        f"Epoch {epoch}/{epochs} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Test Loss: {val_loss:.4f}, Test Acc: {val_acc:.4f}",
        flush=True
    )

elapsed_min = (time.time() - start_time) / 60.0
print(f"\nCompleted {arch.upper()} in {elapsed_min:.1f} minutes.")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

weights_path = f"{arch}_oct2017.pth"
torch.save(model.state_dict(), weights_path)
files.download(weights_path)

del model, optimizer
torch.cuda.empty_cache()

Training DENSENET121...
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 142MB/s]


Epoch 1/2 | Train Loss: 0.1242, Train Acc: 0.9587 | Test Loss: 0.1155, Test Acc: 0.9620
Epoch 2/2 | Train Loss: 0.0829, Train Acc: 0.9719 | Test Loss: 0.1234, Test Acc: 0.9550

Completed DENSENET121 in 66.3 minutes.

Classification Report:
              precision    recall  f1-score   support

         CNV     0.8586    0.9960    0.9222       250
         DME     0.9880    0.9840    0.9860       250
      DRUSEN     1.0000    0.8480    0.9177       250
      NORMAL     0.9960    0.9920    0.9940       250

    accuracy                         0.9550      1000
   macro avg     0.9606    0.9550    0.9550      1000
weighted avg     0.9606    0.9550    0.9550      1000

Confusion Matrix:
 [[249   1   0   0]
 [  4 246   0   0]
 [ 37   0 212   1]
 [  0   2   0 248]]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import time
from google.colab import files

arch = "vgg16"
epochs = 2

print(f"Training {arch.upper()}...")
model = build_transfer_model(arch, num_classes=len(class_names), freeze_backbone=False)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

start_time = time.time()
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, y_true, y_pred = evaluate_model(model, test_loader, criterion, device)
    print(
        f"Epoch {epoch}/{epochs} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Test Loss: {val_loss:.4f}, Test Acc: {val_acc:.4f}",
        flush=True
    )

elapsed_min = (time.time() - start_time) / 60.0
print(f"\nCompleted {arch.upper()} in {elapsed_min:.1f} minutes.")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

weights_path = f"{arch}_oct2017.pth"
torch.save(model.state_dict(), weights_path)
files.download(weights_path)

del model, optimizer
torch.cuda.empty_cache()

Training VGG16...
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:09<00:00, 60.3MB/s]
